In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset, DatasetDict
import pandas as pd

# # Load datasets
# toxicity_data = load_dataset("toxicity_dataset")
# hate_speech_data = load_dataset("hate_speech_dataset")
# fake_news_data = load_dataset("fake_news_dataset")

# Load datasets
combinedData = pd.read_csv("/home/s2shsinh/TWON_Metrics/dataset/merged_dataset_4jan.csv")
# Split into subsets
fake_news_data = combinedData.iloc[0:5000]        # Rows 0 to 4999 (first 5000 rows)
hate_speech_data = combinedData.iloc[5000:10000]  # Rows 5000 to 9999 (next 5000 rows)
toxicity_data = combinedData.iloc[10000:15000]    # Rows 10000 to 14999 (final 5000 rows)

# Print info for confirmation
print("Fake News Data Shape:", fake_news_data.shape)
print("Hate Speech Data Shape:", hate_speech_data.shape)
print("Toxicity Data Shape:", toxicity_data.shape)

def preprocess_data(row, task_name):
    row['text'] = f"[TASK: {task_name}] " + str(row['text'])
    return row

# Apply preprocessing to each DataFrame
toxicity_data = toxicity_data.apply(lambda row: preprocess_data(row, "Toxicity"), axis=1)
hate_speech_data = hate_speech_data.apply(lambda row: preprocess_data(row, "HateSpeech"), axis=1)
fake_news_data = fake_news_data.apply(lambda row: preprocess_data(row, "FakeNews"), axis=1)

Fake News Data Shape: (5000, 4)
Hate Speech Data Shape: (5000, 4)
Toxicity Data Shape: (5000, 4)


In [14]:
toxicity_data

,text,is_fake,is_hate_speech,is_toxic
10000,[TASK: Toxicity] untaten flüchtlingen moslems ...,-1,-1,0
10001,[TASK: Toxicity] regenwald akkord abgeholzt in...,-1,-1,0
10002,[TASK: Toxicity] geht darum makler rein zieht,-1,-1,0
10003,[TASK: Toxicity] huhhhhh fritz jagst richtig a...,-1,-1,0
10004,[TASK: Toxicity] zensur deutschland werdet änd...,-1,-1,0
...,...,...,...,...
14995,[TASK: Toxicity] jung schwul behindert kriteri...,-1,-1,1
14996,[TASK: Toxicity] bissl einseitige sichtweise i...,-1,-1,1
14997,[TASK: Toxicity] hauptschulabschlussafdwählend...,-1,-1,1
14998,[TASK: Toxicity] sehen kreise kultur politik c...,-1,-1,1


In [15]:
from sklearn.model_selection import train_test_split

# Split each dataset into train and test sets
toxicity_train, toxicity_test = train_test_split(toxicity_data, test_size=0.2, random_state=42)
hate_speech_train, hate_speech_test = train_test_split(hate_speech_data, test_size=0.2, random_state=42)
fake_news_train, fake_news_test = train_test_split(fake_news_data, test_size=0.2, random_state=42)

# Print info for confirmation
print("Toxicity Train Shape:", toxicity_train.shape, "Test Shape:", toxicity_test.shape)
print("Hate Speech Train Shape:", hate_speech_train.shape, "Test Shape:", hate_speech_test.shape)
print("Fake News Train Shape:", fake_news_train.shape, "Test Shape:", fake_news_test.shape)

Toxicity Train Shape: (4000, 4) Test Shape: (1000, 4)
Hate Speech Train Shape: (4000, 4) Test Shape: (1000, 4)
Fake News Train Shape: (4000, 4) Test Shape: (1000, 4)


In [16]:
from datasets import Dataset, DatasetDict, concatenate_datasets

# Convert pandas DataFrames into Hugging Face Datasets
toxicity_train_hf = Dataset.from_pandas(toxicity_train)
toxicity_test_hf = Dataset.from_pandas(toxicity_test)
hate_speech_train_hf = Dataset.from_pandas(hate_speech_train)
hate_speech_test_hf = Dataset.from_pandas(hate_speech_test)
fake_news_train_hf = Dataset.from_pandas(fake_news_train)
fake_news_test_hf = Dataset.from_pandas(fake_news_test)

# Concatenate datasets for training and testing using concatenate_datasets
train_data = concatenate_datasets([toxicity_train_hf, hate_speech_train_hf, fake_news_train_hf])
test_data = concatenate_datasets([toxicity_test_hf, hate_speech_test_hf, fake_news_test_hf])

# Combine into a DatasetDict
combined_data = DatasetDict({
    "train": train_data,
    "test": test_data
})

# Print to confirm the data is combined
print("Combined Training Data Shape:", combined_data["train"].num_rows)
print("Combined Testing Data Shape:", combined_data["test"].num_rows)

Combined Training Data Shape: 12000
Combined Testing Data Shape: 3000


In [17]:

import os
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

81

In [18]:

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [19]:
import torch
# Set the CUDA device
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Check if GPU is available
# device = "cpu"
# print(f"Using device: {device}")

In [20]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
def compute_metrics(pred):
    predictions, labels = pred
    predictions = predictions.argmax(axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


In [21]:

# Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize_data(example):
    return tokenizer(example['text'], truncation=True, padding="max_length")

combined_data = combined_data.map(tokenize_data, batched=True)
# Map labels
label_mapping = {"FakeNews": 0, "HateSpeech": 1, "Toxicity": 2}

def map_labels(example):
    task_name = example["text"].split("[TASK: ")[1].split("]")[0]
    example["labels"] = label_mapping[task_name]
    return example

combined_data = combined_data.map(map_labels)


model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3).to(device)

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
from collections import Counter
print("Training Class Distribution:", Counter(combined_data["train"]["labels"]))
print("Test Class Distribution:", Counter(combined_data["test"]["labels"]))

Training Class Distribution: Counter({2: 4000, 1: 4000, 0: 4000})
Test Class Distribution: Counter({2: 1000, 1: 1000, 0: 1000})


In [24]:
train_texts = set(combined_data["train"]["text"])
test_texts = set(combined_data["test"]["text"])
overlap = train_texts.intersection(test_texts)
print(f"Overlap between train and test: {len(overlap)} samples")

Overlap between train and test: 157 samples


In [ ]:



# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    gradient_accumulation_steps=2,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=combined_data["train"],
    eval_dataset=combined_data["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
results = trainer.evaluate()
print(results)